In [2]:

import logging
import os

from dotenv import load_dotenv
load_dotenv(override=True)
logger = logging.getLogger(__name__)
def _gather_aws_context() -> str:
        try:
            import boto3
            from botocore.exceptions import BotoCoreError, ClientError

            session = boto3.session.Session()
            region = session.region_name or os.getenv("AWS_DEFAULT_REGION") or os.getenv("AWS_REGION") or ""

            lines = ["AWS ACCOUNT GROUNDING (live, fetched at pipeline start — treat as ground truth):"]

            try:
                sts = session.client("sts", region_name=region or None)
                identity = sts.get_caller_identity()
                lines.append(f"  Account ID : {identity.get('Account')}")
                lines.append(f"  Caller ARN : {identity.get('Arn')}")
            except (BotoCoreError, ClientError) as exc:
                logger.warning("aws_context.sts_failed: %s", exc)
                lines.append("  Account ID : (unavailable — could not call sts:GetCallerIdentity)")

            lines.append(f"  Region     : {region or '(not set — will rely on provider config)'}")

            try:
                ec2 = session.client("ec2", region_name=region or None)
                vpcs = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])
                default_vpc = (vpcs.get("Vpcs") or [{}])[0].get("VpcId")
                if default_vpc:
                    lines.append(f"  Default VPC: {default_vpc}")
                    subnets = ec2.describe_subnets(
                        Filters=[{"Name": "vpc-id", "Values": [default_vpc]}]
                    )
                    subnet_ids = [s["SubnetId"] for s in subnets.get("Subnets", [])]
                    if subnet_ids:
                        lines.append(f"  Default VPC subnets: {', '.join(subnet_ids)}")
                else:
                    lines.append("  Default VPC: (none found in this account/region)")

                # Fetch available Availability Zones
                azs = ec2.describe_availability_zones()
                az_names = [az["ZoneName"] for az in azs.get("AvailabilityZones", []) if az["State"] == "available"]
                if az_names:
                    lines.append(f"  Available AZs: {', '.join(az_names)}")

                # Fetch existing Key Pairs
                key_pairs = ec2.describe_key_pairs()
                kp_names = [kp["KeyName"] for kp in key_pairs.get("KeyPairs", [])]
                if kp_names:
                    lines.append(f"  Existing Key Pairs: {', '.join(kp_names)}")
                else:
                    lines.append("  Existing Key Pairs: (none found, you must generate one if needed)")
                # Fetch existing Security Groups in Default VPC
                if default_vpc:
                    sgs = ec2.describe_security_groups(Filters=[{"Name": "vpc-id", "Values": [default_vpc]}])
                    sg_info = [f"{sg['GroupName']} ({sg['GroupId']})" for sg in sgs.get("SecurityGroups", [])]
                    if sg_info:
                        lines.append(f"  Existing Security Groups: {', '.join(sg_info)}")
                    else:
                        lines.append("  Existing Security Groups: (none found)")

            except (BotoCoreError, ClientError) as exc:
                logger.warning("aws_context.ec2_failed: %s", exc)

            try:
                route53 = session.client("route53", region_name=region or None)
                zones = route53.list_hosted_zones()
                zone_info = [f"{z['Name']} (ID: {z['Id']})" for z in zones.get("HostedZones", []) if not z["Config"]["PrivateZone"]]
                if zone_info:
                    lines.append(f"  Public Route53 Zones: {', '.join(zone_info)}")
                else:
                    lines.append("  Public Route53 Zones: (none found)")
            except (BotoCoreError, ClientError) as exc:
                logger.warning("aws_context.route53_failed: %s", exc)

            lines.append(
                "\nIf an ID is provided in the context above (VPC, Subnets, Security Groups, Key Pairs, Route53 Zones), hardcode it directly in your Terraform code to avoid data source filter errors. "
                "ONLY use Terraform data sources (e.g. data \"aws_vpc\") if the required resource is marked as '(none found)' or is missing from the context.\n"
                "\nTERRAFORM GOLDEN RULES:\n"
                "1. S3 Buckets: Names must be globally unique. Always use random_id or random_pet to append a suffix to bucket names.\n"
                "2. IAM Roles/Policies: Always use name_prefix instead of name to avoid conflicts with existing roles.\n"
                "3. EC2/RDS Security Groups: Prefer using existing security groups if they match your needs, or use name_prefix when creating new ones.\n"
                "4. Circular Dependencies: Never make a Security Group depend on an EC2 instance's IP if the EC2 instance also depends on that Security Group.\n"
                "5. Hardcoding: Hardcode environment IDs (like VPCs) *only* if they are provided in the context above. NEVER hardcode full ARNs or Regions (use data.aws_caller_identity.current and data.aws_region.current instead).\n"
                "6. Stateful Resources: For databases (RDS, DynamoDB) and storage (S3), always set lifecycle { prevent_destroy = true } unless instructed otherwise.\n"
                "7. Provider Version: Use the required_providers block to specify hashicorp/aws version ~> 5.0 to ensure modern syntax is supported."
            )
            return "\n".join(lines)
        except ImportError:
            logger.info("aws_context.boto3_unavailable — skipping live AWS grounding")
            return ""
        except Exception as exc:
            logger.warning("aws_context.failed: %s", exc)
            return ""
        
result = _gather_aws_context()
print(result)

AWS ACCOUNT GROUNDING (live, fetched at pipeline start — treat as ground truth):
  Account ID : 827295473120
  Caller ARN : arn:aws:iam::827295473120:user/observation_agent
  Region     : us-east-1
  Default VPC: vpc-092ce2cf644fddc27
  Default VPC subnets: subnet-0991ae1a54e1313bc, subnet-0824abd82167b04da, subnet-0b7bc0c382430c442, subnet-0ffa581cb13e5e816, subnet-09e14f075dfbf6445, subnet-0e36ffd51a5d57df7
  Available AZs: us-east-1a, us-east-1b, us-east-1c, us-east-1d, us-east-1e, us-east-1f
  Existing Key Pairs: Chandra 05-04, Email-agent, mahesh-key, Chandra-public-key, Chandra v1
  Existing Security Groups: launch-wizard-1 (sg-0f1a7667b884947ff), launch-wizard-2 (sg-0aa4f7618c1bac288), rel-001-postgres-sg (sg-04c5590f9cb4a1946), default (sg-084543c18876a4b99), rds-postgres-access (sg-0bbca4a13e1742259)
  Public Route53 Zones: (none found)

If an ID is provided in the context above (VPC, Subnets, Security Groups, Key Pairs, Route53 Zones), hardcode it directly in your Terraform c

In [6]:
import boto3
def list_all_services_with_quotas(region_name: str = "us-east-1"):
    client = boto3.client('service-quotas', region_name=region_name)
    paginator = client.get_paginator('list_services')
    
    services = []
    for page in paginator.paginate():
        for svc in page.get('Services', []):
            services.append(f"{svc['ServiceName']} -> ServiceCode: '{svc['ServiceCode']}'")
            
    return services

if __name__ == "__main__":
    result = list_all_services_with_quotas()
    for svc in result:
        print(svc)

AWS Cloud Map -> ServiceCode: 'AWSCloudMap'
Access Analyzer -> ServiceCode: 'access-analyzer'
AWS Account Management -> ServiceCode: 'account'
AWS Certificate Manager (ACM) -> ServiceCode: 'acm'
AWS Private Certificate Authority -> ServiceCode: 'acm-pca'
AWS Compute Optimizer Automation -> ServiceCode: 'aco-automation'
AWS DevOps Agent -> ServiceCode: 'aidevops'
Amazon AI Operations -> ServiceCode: 'aiops'
Amazon Managed Workflows for Apache Airflow -> ServiceCode: 'airflow'
Amazon MWAA Serverless (airflow-serverless) -> ServiceCode: 'airflow-serverless'
AWS Amplify -> ServiceCode: 'amplify'
Amplify UI Builder -> ServiceCode: 'amplifyuibuilder'
Amazon OpenSearch Serverless -> ServiceCode: 'aoss'
Amazon API Gateway -> ServiceCode: 'apigateway'
Amazon Connect Application Integrations -> ServiceCode: 'app-integrations'
AWS AppConfig -> ServiceCode: 'appconfig'
AWS AppFabric -> ServiceCode: 'appfabric'
Amazon AppFlow -> ServiceCode: 'appflow'
Application Auto Scaling -> ServiceCode: 'appli

In [9]:
import boto3

def get_all_service_quotas(service_code: str, region_name: str = "us-east-1"):
    """
    Fetches a list of all available Service Quotas for a given AWS service.
    
    Args:
        service_code (str): The service code (e.g., 'ec2', 'vpc', 'lambda', 's3').
        region_name (str): The AWS region to check.
        
    Returns:
        list: A list of dictionaries containing quota details.
    """
    client = boto3.client('service-quotas', region_name=region_name)
    quotas = []
    
    # Initialize the paginator since some services have hundreds of quotas
    paginator = client.get_paginator('list_service_quotas')
    page_iterator = paginator.paginate(ServiceCode=service_code)
    
    try:
        for page in page_iterator:
            for quota in page.get('Quotas', []):
                quotas.append({
                    "QuotaName": quota.get("QuotaName"),
                    "QuotaCode": quota.get("QuotaCode"),
                    "Value": quota.get("Value"),
                    "Adjustable": quota.get("Adjustable")
                })
                
        print(f"Successfully fetched {len(quotas)} quotas for {service_code}.")
        return quotas

    except Exception as e:
        print(f"Failed to fetch quotas for '{service_code}': {e}")
        return []

# --- Example Usage ---
if __name__ == "__main__":
    # Fetch all quotas for AWS Lambda
    lambda_quotas = get_all_service_quotas("ec2")
    
    for q in lambda_quotas[:5]: # Print the first 5 quotas
        print(f"- QuotaName: {q['QuotaName']}, QuotaCode: {q['QuotaCode']}, Value: {q['Value']}, Adjustable: {q['Adjustable']}")

Successfully fetched 1761 quotas for ec2.
- QuotaName: ModifyVerifiedAccessEndpoint request bucket maximum capacity, QuotaCode: L-2481C19E, Value: 20.0, Adjustable: False
- QuotaName: DeleteCustomerGateway request bucket maximum capacity, QuotaCode: L-9B6899DE, Value: 50.0, Adjustable: False
- QuotaName: ModifySnapshotTier request bucket refill rate, QuotaCode: L-2394664B, Value: 5.0, Adjustable: False
- QuotaName: GetIpamPrefixListResolverVersionEntries request bucket refill rate, QuotaCode: L-7186273C, Value: 20.0, Adjustable: False
- QuotaName: DescribeTransitGatewayMulticastDomains request bucket refill rate, QuotaCode: L-FB79A78F, Value: 20.0, Adjustable: False


In [10]:
import boto3
import urllib.request
import re
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("DynamicGrounding")

def test_gather_docs_and_quotas(services, resources):
    quotas_context = ""
    if services:
        quotas_context += "\nSERVICE QUOTAS (from dynamic lookup):\n"
        try:
            client = boto3.client('service-quotas', region_name='us-east-1')
            for svc in services:
                try:
                    res = client.list_service_quotas(ServiceCode=svc.lower(), MaxResults=20)
                    if res.get('Quotas'):
                        quotas_context += f"--- {svc.upper()} Quotas ---\n"
                        for q in res['Quotas']:
                            quotas_context += f"- {q.get('QuotaName')}: {q.get('Value')}\n"
                except Exception as e:
                    logger.warning("Failed to fetch quotas for %s: %s", svc, e)
        except Exception as e:
            logger.warning("Boto3 service-quotas error: %s", e)
            
    docs_context = ""
    if resources:
        docs_context += "\nTERRAFORM DOCUMENTATION (Argument References):\n"
        for res_type in resources:
            if not res_type.startswith("aws_"):
                continue
            short_name = res_type[4:]
            url = f"https://raw.githubusercontent.com/hashicorp/terraform-provider-aws/main/website/docs/r/{short_name}.html.markdown"
            try:
                req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=5) as response:
                    markdown = response.read().decode('utf-8')
                    
                match = re.search(r'## Argument Reference(.*?)(?:##|\Z)', markdown, re.DOTALL | re.IGNORECASE)
                if match:
                    snippet = match.group(1).strip()
                    docs_context += f"\n--- {res_type} Argument Reference ---\n{snippet[:1500]}...\n"
                else:
                    docs_context += f"\n--- {res_type} ---\nNo Argument Reference found in docs.\n"
            except Exception as e:
                logger.warning("Failed to fetch TF docs for %s at %s: %s", res_type, url, e)

    logger.info("Dynamic Grounding: Analyzer identified services=%s and resources=%s", services, resources)
    logger.info("Dynamic Grounding: Fetched quotas for %d services, docs for %d resources.", len(services), len(resources))
    
    return {
        "service_quotas_context": quotas_context,
        "terraform_docs": docs_context
    }

if __name__ == "__main__":
    # Simulate Analyze Agent output
    services = ["lambda"]
    resources = ["aws_lambda_function"]
    
    result = test_gather_docs_and_quotas(services, resources)
    
    print("\n================== OUTPUT ==================\n")
    print(result["service_quotas_context"])
    print(result["terraform_docs"])

INFO:DynamicGrounding:Dynamic Grounding: Analyzer identified services=['lambda'] and resources=['aws_lambda_function']
INFO:DynamicGrounding:Dynamic Grounding: Fetched quotas for 1 services, docs for 1 resources.



================== OUTPUT ==================


SERVICE QUOTAS (from dynamic lookup):
--- LAMBDA Quotas ---
- Number of concurrent MicroVM image builds: 10.0
- Rate of capacity provider write API requests: 1.0
- Rate of SendDurableExecutionCallbackSuccess API requests: 300.0
- Max Execution Duration of a MicroVM (in Hours): 8.0
- Burst rate of ResumeMicrovm API requests: 5.0
- Rate of GetDurableExecutionHistory API requests: 15.0


TERRAFORM DOCUMENTATION (Argument References):

--- aws_lambda_function Argument Reference ---
The following arguments are required:

* `function_name` - (Required) Unique name for your Lambda Function.
* `role` - (Required) ARN of the function's execution role. The role provides the function's identity and access to AWS services and resources.

The following arguments are optional:

* `architectures` - (Optional) Instruction set architecture for your Lambda function. Valid values are `["x86_64"]` and `["arm64"]`. Default is `["x86_64"]`. Removing this attri

In [4]:
import json
import logging
import os
import re
import urllib.request
from typing import Dict, List, Optional

from langchain_mcp_adapters.client import MultiServerMCPClient

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("DynamicGrounding.MCP")

os.environ.setdefault("AWS_REGION", "us-east-1")

SERVER_CONFIG = {
    "terraform": {
        "command": "docker",
        "args": ["run", "-i", "--rm", "hashicorp/terraform-mcp-server"],
        "transport": "stdio",
    },
    "aws_api": {
        "command": "uvx",
        "args": ["awslabs.aws-api-mcp-server@latest"],
        "env": {"AWS_REGION": os.environ["AWS_REGION"]},
        "transport": "stdio",
    },
}

client = MultiServerMCPClient(SERVER_CONFIG)
print("client ready")

client ready


In [5]:
all_tools = {}
for server_name in SERVER_CONFIG:
    tools = await client.get_tools(server_name=server_name)
    all_tools[server_name] = {t.name: t for t in tools}
    print(f"\n=== {server_name} ({len(tools)} tools) ===")
    for t in tools:
        print(f"- {t.name}: {(t.description or '').strip()[:150]}")

UnsupportedOperation: fileno

In [ ]:
QUOTAS_TOOL_NAME = "call_aws"             # <- replace if Cell 3 showed a different name
DOCS_TOOL_NAME = "resolveProviderDocID"   # <- replace if Cell 3 showed a different name
async def fetch_service_quotas_via_mcp(svc: str) -> Optional[str]:
    tool = all_tools.get("aws_api", {}).get(QUOTAS_TOOL_NAME)
    if tool is None:
        logger.warning("quotas tool '%s' not found", QUOTAS_TOOL_NAME)
        return None
    region = os.environ["AWS_REGION"]
    try:
        raw = await tool.ainvoke({
            "cli_command": f"aws service-quotas list-service-quotas --service-code {svc.lower()} --max-results 20 --region {region}"
        })
        parsed = json.loads(raw) if isinstance(raw, str) else raw
        quotas = parsed.get("Quotas", []) if isinstance(parsed, dict) else []
        if not quotas:
            return f"--- {svc.upper()} Quotas ---\n(no quotas returned)\n"
        svc_ctx = f"--- {svc.upper()} Quotas ---\n"
        for q in quotas:
            svc_ctx += f"- {q.get('QuotaName')}: {q.get('Value')}\n"
        return svc_ctx
    except Exception as e:
        logger.warning("Failed to fetch quotas for %s via MCP: %s", svc, e)
        return None


async def fetch_terraform_docs_via_mcp(res_type: str) -> Optional[str]:
    tool = all_tools.get("terraform", {}).get(DOCS_TOOL_NAME)
    if tool is None:
        logger.warning("docs tool '%s' not found", DOCS_TOOL_NAME)
        return None
    try:
        raw = await tool.ainvoke({
            "providerName": "aws",
            "providerNamespace": "hashicorp",
            "serviceSlug": res_type,
        })
        text = raw if isinstance(raw, str) else json.dumps(raw, default=str)
        return f"\n--- {res_type} Argument Reference (via MCP) ---\n{text[:1500]}...\n"
    except Exception as e:
        logger.warning("Failed to fetch TF docs for %s via MCP: %s", res_type, e)
        return None

In [ ]:
async def test_gather_docs_and_quotas_mcp(services: List[str], resources: List[str]) -> Dict[str, str]:
    quotas_context = ""
    if services:
        quotas_context += "\nSERVICE QUOTAS (from dynamic lookup, via aws_api MCP):\n"
        for svc in services:
            svc_ctx = await fetch_service_quotas_via_mcp(svc)
            if svc_ctx:
                quotas_context += svc_ctx

    docs_context = ""
    if resources:
        docs_context += "\nTERRAFORM DOCUMENTATION (Argument References, via terraform MCP):\n"
        for res_type in resources:
            if res_type.startswith("aws_"):
                res_ctx = await fetch_terraform_docs_via_mcp(res_type)
                if res_ctx:
                    docs_context += res_ctx

    return {"service_quotas_context": quotas_context, "terraform_docs": docs_context}


services = ["lambda"]
resources = ["aws_lambda_function"]

result = await test_gather_docs_and_quotas_mcp(services, resources)

print("\n================== MCP OUTPUT ==================\n")
print(result["service_quotas_context"])
print(result["terraform_docs"])